### Professional ETL Pipeline Blueprint

* **Standardize Column Names**
  * Correct typos and naming conventions (e.g., `product_name_lenght` $\rightarrow$ `product_name_length`).
* **Convert Data Types**
  * Cast date strings to `datetime` objects.
  * Map numeric columns to their correct data types.
* **Handle Missing Values**
  * Drop, impute, or retain missing data based on discovered business rules.
* **Handle Outliers**
  * Flag and investigate unrealistic variables (e.g., anomalous prices, weights, freight values).
* **Clean Categorical Values**
  * Trim whitespace, normalize text casing, and validate category domains.
* **Create a Clean Relational Dataset**
  * Ensure each source table is isolated, reliable, and production-ready.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

In [2]:
customers = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_order_payments_dataset.csv")
products = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_products_dataset.csv")
reviews = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_order_reviews_dataset.csv")
sellers = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_sellers_dataset.csv")
geolocation = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/olist_geolocation_dataset.csv")
translation = pd.read_csv("c:/Users/Steve/Desktop/datascience-project1/project/data/raw/product_category_name_translation.csv")

In [3]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "products": products,
    "reviews": reviews,
    "sellers": sellers,
    "geolocation": geolocation,
    "translation": translation
}

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

In [5]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### we are gonna check inconsistent date, times, etc. standizing the column names values etc. 

### Validation 1.1
Purchase should happen before Approval

In [6]:
invalid_approval = orders[
    orders["order_approved_at"] < orders["order_purchase_timestamp"]
]
# stored as a list then len() is used to get the number of invalid orders.
print(f"Invalid Orders: {len(invalid_approval)}")

invalid_approval.head()

Invalid Orders: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


Case 1
Invalid Orders: 0

Case 2
Invalid Orders: 8

Now we investigate.

Questions:

Data entry issue?
Timezone issue?
System bug?
Import problem?

Again...

We investigate.
We don't immediately fix.

Why is this important?

Suppose you're asked

"Average payment approval time."

If approvals happen before purchases...

Your KPI becomes nonsense.

### Validation 1.2

Question

Can the carrier receive the package before the order was approved?

Obviously not.


In [7]:
invalid_carrier = orders[
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
]

print(f"Invalid Orders: {len(invalid_carrier)}")

invalid_carrier.head()

Invalid Orders: 1359


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06


### List the first 10 invalid_carrier checkup

In [8]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

In [9]:
invalid_carrier[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_status"
    ]
].head(10)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_status
15,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,delivered
64,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,delivered
199,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,delivered
210,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,delivered
415,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,delivered
481,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,delivered
483,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,delivered
585,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,delivered
615,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,delivered
817,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,delivered


In [10]:
orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date"
    ]
].dtypes

order_purchase_timestamp        datetime64[us]
order_approved_at               datetime64[us]
order_delivered_carrier_date    datetime64[us]
dtype: object

### Check the delay or gap / Measure the difference

In [11]:
delay = (
    invalid_carrier["order_approved_at"]
    - invalid_carrier["order_delivered_carrier_date"]
)

delay.describe()

count                      1359
mean     1 days 00:45:07.153053
std      4 days 19:18:28.055754
min             0 days 00:00:21
25%      0 days 01:24:55.500000
50%             0 days 17:10:04
75%             1 days 01:57:24
max           171 days 05:15:22
dtype: object

### we have suspicious difference between APPROVAL_DATE AND THE DELIVERED_TO_CARRIER


Let's find the worst offenders.

In [12]:
invalid_carrier = invalid_carrier.copy()

invalid_carrier["approval_delay"] = (
    invalid_carrier["order_approved_at"]
    - invalid_carrier["order_delivered_carrier_date"]
)

invalid_carrier.sort_values(
    by="approval_delay",
    ascending=False
)[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "approval_delay",
        "order_status"
    ]
].head(10)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,approval_delay,order_status
25883,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,171 days 05:15:22,delivered
14562,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-04 13:10:23,9 days 08:55:48,delivered
46163,2017-09-01 20:04:28,2017-09-13 22:17:15,2017-09-04 14:05:50,9 days 08:11:25,delivered
98710,2017-09-01 20:28:02,2017-09-13 22:03:51,2017-09-04 18:07:55,9 days 03:55:56,delivered
41592,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,9 days 02:45:45,delivered
11738,2017-09-01 18:45:33,2017-09-13 22:04:39,2017-09-04 20:12:41,9 days 01:51:58,delivered
85393,2017-09-01 20:05:55,2017-09-13 21:58:38,2017-09-04 20:36:58,9 days 01:21:40,delivered
31211,2017-09-01 20:05:42,2017-09-13 22:00:51,2017-09-04 20:49:57,9 days 01:10:54,delivered
55302,2017-09-01 18:49:54,2017-09-13 21:56:08,2017-09-04 20:50:00,9 days 01:06:08,delivered
68315,2017-09-01 20:17:09,2017-09-13 22:03:42,2017-09-05 11:58:18,8 days 10:05:24,delivered


171 DAYS also abnormal also before even purchase how it was sent to carrier 

Let's investigate the pattern

In [13]:
invalid_carrier["order_approved_at"].dt.date.value_counts().head(10)

order_approved_at
2018-07-05    451
2018-04-24    383
2018-08-01     22
2018-07-26     18
2018-07-23     18
2018-07-27     18
2018-02-04     18
2017-08-02     18
2018-06-27     15
2018-07-28     14
Name: count, dtype: int64

We know when approvals happened.

Now we should ask:

When were these orders actually purchased?

For example, let's investigate the biggest group (2018-07-05).

In [14]:

invalid_carrier[
    invalid_carrier["order_approved_at"].dt.date == pd.Timestamp("2018-07-05").date()
][
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date"
    ]
].sort_values("order_purchase_timestamp").head(20)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date
71206,2018-06-29 00:02:57,2018-07-05 16:08:15,2018-07-04 14:15:00
64496,2018-06-29 05:26:29,2018-07-05 16:08:11,2018-07-03 19:29:00
81235,2018-06-29 08:12:42,2018-07-05 16:02:24,2018-07-04 12:57:00
1913,2018-06-29 11:19:19,2018-07-05 16:04:10,2018-07-04 13:55:00
70100,2018-06-29 14:24:39,2018-07-05 16:06:14,2018-07-03 13:22:00
92793,2018-06-29 18:20:57,2018-07-05 16:04:53,2018-07-03 13:08:00
76217,2018-06-29 18:27:05,2018-07-05 16:11:45,2018-07-03 19:01:00
79954,2018-06-29 18:52:27,2018-07-05 16:01:00,2018-07-04 06:49:00
71269,2018-06-29 19:15:06,2018-07-05 16:02:57,2018-07-03 15:14:00
63285,2018-06-29 21:56:38,2018-07-05 16:04:23,2018-07-04 12:24:00


### Validation 2 — Delivery After Purchase

In [15]:
invalid_delivery = orders[
    orders["order_delivered_customer_date"] <
    orders["order_purchase_timestamp"]
]

print(f"Invalid Deliveries: {len(invalid_delivery)}")
invalid_delivery.head()

Invalid Deliveries: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


### Validation 3 — Delivery After Carrier Pickup

In [17]:
invalid_delivery_sequence = orders[
    orders["order_delivered_customer_date"] <
    orders["order_delivered_carrier_date"]
]

print(
    f"Invalid Delivery Sequence: {len(invalid_delivery_sequence)}"
)

invalid_delivery_sequence.head(10)

Invalid Delivery Sequence: 23


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,6cd7106899e59a1fbd0622d5f1efedf4,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,9859dd92e872dbaa60ca3cd5f0d7ad07,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,01c843a2c0600def0b7693dba47af460,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,3947a361301f2ff0f3223159a0f2701c,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,63be4feff10a0b1d85f2cfbf10df9754,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


### Estimated vs Actual Delivery (Business KPI)

In [18]:
late_orders = orders[
    orders["order_delivered_customer_date"] >
    orders["order_estimated_delivery_date"]
]

print(f"Late Deliveries: {len(late_orders)}")

Late Deliveries: 7827


In [19]:
late_orders["delay"] = (
    late_orders["order_delivered_customer_date"]
    -
    late_orders["order_estimated_delivery_date"]
)

late_orders["delay"].describe()

count                       7827
mean      9 days 13:15:10.612878
std      13 days 22:50:26.879558
min              0 days 00:03:36
25%       1 days 20:48:45.500000
50%              5 days 19:23:22
75%      11 days 19:44:21.500000
max            188 days 23:24:07
Name: delay, dtype: object

### Validation 5 — Review Score Range

In [22]:
invalid_reviews = reviews[
    ~reviews["review_score"].between(1,5)
]

print(len(invalid_reviews))
invalid_reviews.head(10)

0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


### Validation 6 — Payment Amount (Business rule)
Payment should never be negative.



In [24]:
negative_payment = payments[
    payments["payment_value"] < 0
]
print(f"Negative Payments: {len(negative_payment)}")

Negative Payments: 0


#### Validation 7 - Freight

In [26]:

negative_freight = order_items[
    order_items["freight_value"] < 0
]
print(f"Negative Freight Charges: {len(negative_freight)}")

Negative Freight Charges: 0



#### Validation 8 — Product Price


In [27]:
invalid_price = order_items[
    order_items["price"] <= 0
]

print(len(invalid_price))

0


#### Validation 9 — Installments

In [30]:
invalid_installments = payments[
    payments["payment_installments"] <= 0
]

print(f"Invalid Installments: {len(invalid_installments)}")
invalid_installments.head(5)

Invalid Installments: 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


#### Validation 10 — Duplicate Order IDs (Business Rule)
Every order should appear exactly once in the Orders table.

In [31]:

orders["order_id"].duplicated().sum()

np.int64(0)

### Validation 11 — Status Validation
Check available values.

In [32]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64